# 04. mRMR Feature Selection

- 목적: `data_vif.csv`에서 **train 데이터만 사용해 mRMR 변수선택**을 수행합니다.
- mRMR = minimum Redundancy Maximum Relevance
- 핵심 로직: `Risk_Label`과 관련성은 높은 변수, 이미 선택된 변수들과 중복성은 낮은 변수를 순차적으로 선택합니다.
- `valid/test`는 변수선택에 사용하지 않고, 이후 최종 모델 튜닝/평가용으로만 분리합니다.
- 마지막 셀에서 선택된 피처 목록만 `print()`합니다.


In [1]:
# =========================
# 0. Import & settings
# =========================
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [2]:
# =========================
# 1. User settings
# =========================
DATA_PATH = Path(r"../../data/processed/data_vif.csv")
TARGET = "Risk_Label"

TRAIN_RATIO = 0.45
VALID_RATIO = 0.35
TEST_RATIO  = 0.20

TOP_K = 12          # 최종 선택할 변수 개수. 필요하면 여기만 수정.
RANDOM_STATE = 1


In [3]:
# =========================
# 2. Load data & target conversion
# =========================
df = pd.read_csv(DATA_PATH)

print("=" * 80)
print("Loaded data")
print("=" * 80)
print("path:", DATA_PATH)
print("shape:", df.shape)
print("columns:", list(df.columns))

if TARGET not in df.columns:
    raise ValueError(f"'{TARGET}' 컬럼이 없습니다.")

# Date가 있으면 시간순 정렬
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

# Risk_Label 안전 변환: 0/1, 0.0/1.0, Low Risk/High Risk 모두 대응
label_text = df[TARGET].astype("string").str.strip()

label_map = {
    "Low Risk": 0,
    "Low risk": 0,
    "low risk": 0,
    "LOW RISK": 0,
    "High Risk": 1,
    "High risk": 1,
    "high risk": 1,
    "HIGH RISK": 1,
    "0": 0,
    "0.0": 0,
    "1": 1,
    "1.0": 1,
}

mapped_label = label_text.map(label_map)
numeric_label = pd.to_numeric(df[TARGET], errors="coerce")
df[TARGET] = mapped_label.fillna(numeric_label)

if df[TARGET].isna().any():
    bad_values = label_text[df[TARGET].isna()].unique()
    raise ValueError(f"Risk_Label 변환 실패 값이 있습니다: {bad_values}")

df[TARGET] = df[TARGET].astype(int)

print("\nRisk_Label distribution")
print(df[TARGET].value_counts().sort_index())


Loaded data
path: ..\..\data\processed\data_vif.csv
shape: (4108, 30)
columns: ['Date', 'KODEX 200_Premium', 'KOSDAQ_return(%)', 'NASDAQ_return(%)', 'Brent Crude Oil_return(%)', 'Gold Spot_return(%)', 'USD/KRW_change(%)', 'KOSPI 200 Volume', 'return(%)', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200 lagged_2_return(%)', 'VKOSPI_Close', 'VKOSPI_Change(%)', 'KOSPI 200_RSI14', 'KOSPI 200_DMI14', 'KOSPI 200_ADX14', 'KOSPI 200_BB_width', 'KOSPI 200_PPO', 'KOSPI 200_PPO_Hist', 'KOSPI 200_OG', 'GJR_VaR_5_t1', 'Signal1_Buy', 'Signal1_Sell', 'Signal2_Buy', 'Signal2_Sell', 'Signal3_Buy', 'Signal3_Sell', 'Signal4_Buy', 'Signal4_Sell', 'Risk_Label']

Risk_Label distribution
Risk_Label
0    3617
1     491
Name: count, dtype: int64


In [4]:
# =========================
# 3. X / y 구성
# =========================
drop_cols = [TARGET]
if "Date" in df.columns:
    drop_cols.append("Date")

X = df.drop(columns=drop_cols)
y = df[TARGET].copy()

# 문자형 변수가 남아 있으면 더미화
object_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

if len(object_cols) > 0:
    print("Categorical columns detected and one-hot encoded:")
    print(object_cols)
    X = pd.get_dummies(X, columns=object_cols, drop_first=False, dtype=int)

# 숫자형 변환
X = X.apply(pd.to_numeric, errors="coerce")

print("Feature matrix shape after encoding:", X.shape)
display(X.head())


Feature matrix shape after encoding: (4108, 28)


,KODEX 200_Premium,KOSDAQ_return(%),NASDAQ_return(%),Brent Crude Oil_return(%),Gold Spot_return(%),USD/KRW_change(%),KOSPI 200 Volume,return(%),KOSPI 200 lagged_1_return(%),KOSPI 200 lagged_2_return(%),VKOSPI_Close,VKOSPI_Change(%),KOSPI 200_RSI14,KOSPI 200_DMI14,KOSPI 200_ADX14,KOSPI 200_BB_width,KOSPI 200_PPO,KOSPI 200_PPO_Hist,KOSPI 200_OG,GJR_VaR_5_t1,Signal1_Buy,Signal1_Sell,Signal2_Buy,Signal2_Sell,Signal3_Buy,Signal3_Sell,Signal4_Buy,Signal4_Sell
0,-0.20,-2.796416,0.157320,0.0,-1.362586,-0.549095,251600000.0,-0.233192,0.303254,-0.477796,35.49,-2.10,63.873746,14.647201,28.067103,0.131276,3.548332,0.132689,1.339314,-2.736404,0,0,0,0,0,0,0,0
1,-0.34,1.668519,-3.953850,0.0,2.234472,0.128139,183300000.0,0.564563,-0.233192,0.303254,36.15,1.86,76.439644,12.666581,27.802004,0.131521,3.485846,0.056162,0.712077,-2.640948,0,0,0,0,0,0,0,0
2,-0.14,1.061549,2.191930,0.0,-0.553958,1.998728,204200000.0,-0.197523,0.564563,-0.233192,36.40,0.69,74.958299,9.561273,27.172803,0.112261,3.379426,-0.040207,-2.159026,-2.553087,0,0,0,0,0,0,0,0
3,-0.19,2.524236,0.137996,0.0,1.093648,-0.570187,291400000.0,1.408955,-0.197523,0.564563,35.01,-3.82,74.084326,13.493950,27.080292,0.090814,3.371206,-0.038741,0.767616,-2.469280,0,0,0,0,0,0,0,0
4,-0.17,0.818378,0.369276,0.0,1.568707,-0.970084,277400000.0,0.992765,1.408955,-0.197523,33.39,-4.63,69.951956,15.409494,27.250038,0.079146,3.405654,-0.003434,0.848630,-2.386329,0,0,0,0,0,1,0,0


In [5]:
# =========================
# 4. Chronological split
#    mRMR 선택은 train에서만 수행
# =========================
n_total = len(df)
train_end = int(n_total * TRAIN_RATIO)
valid_end = int(n_total * (TRAIN_RATIO + VALID_RATIO))

X_train = X.iloc[:train_end].reset_index(drop=True)
y_train = y.iloc[:train_end].reset_index(drop=True)

X_valid = X.iloc[train_end:valid_end].reset_index(drop=True)
y_valid = y.iloc[train_end:valid_end].reset_index(drop=True)

X_test = X.iloc[valid_end:].reset_index(drop=True)
y_test = y.iloc[valid_end:].reset_index(drop=True)

print("=" * 80)
print("Split size check")
print("=" * 80)
print(f"total : {n_total}")
print(f"train : {len(X_train)}")
print(f"valid : {len(X_valid)}")
print(f"test  : {len(X_test)}")

print("\nTrain class distribution")
print(y_train.value_counts().sort_index())

print("\nValid class distribution")
print(y_valid.value_counts().sort_index())

print("\nTest class distribution")
print(y_test.value_counts().sort_index())


Split size check
total : 4108
train : 1848
valid : 1438
test  : 822

Train class distribution
Risk_Label
0    1638
1     210
Name: count, dtype: int64

Valid class distribution
Risk_Label
0    1256
1     182
Name: count, dtype: int64

Test class distribution
Risk_Label
0    723
1     99
Name: count, dtype: int64


In [6]:
# =========================
# 5. Missing values & constant columns
#    train 기준 median만 사용
# =========================
train_median = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_median)
X_valid = X_valid.fillna(train_median)
X_test  = X_test.fillna(train_median)

# 전체가 결측이어서 train median도 NaN인 컬럼 제거
nan_cols = X_train.columns[X_train.isna().any()].tolist()
if len(nan_cols) > 0:
    print("Columns removed because train median is NaN:")
    print(nan_cols)
    X_train = X_train.drop(columns=nan_cols)
    X_valid = X_valid.drop(columns=nan_cols)
    X_test  = X_test.drop(columns=nan_cols)

# 상수 컬럼 제거
constant_cols = X_train.columns[X_train.nunique(dropna=True) <= 1].tolist()
if len(constant_cols) > 0:
    print("Constant columns removed:")
    print(constant_cols)
    X_train = X_train.drop(columns=constant_cols)
    X_valid = X_valid.drop(columns=constant_cols)
    X_test  = X_test.drop(columns=constant_cols)

feature_names = X_train.columns.tolist()

if TOP_K > len(feature_names):
    raise ValueError(f"TOP_K={TOP_K}인데 사용 가능한 변수는 {len(feature_names)}개뿐입니다.")

print("Available features:", len(feature_names))
print(feature_names)


Available features: 28
['KODEX 200_Premium', 'KOSDAQ_return(%)', 'NASDAQ_return(%)', 'Brent Crude Oil_return(%)', 'Gold Spot_return(%)', 'USD/KRW_change(%)', 'KOSPI 200 Volume', 'return(%)', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200 lagged_2_return(%)', 'VKOSPI_Close', 'VKOSPI_Change(%)', 'KOSPI 200_RSI14', 'KOSPI 200_DMI14', 'KOSPI 200_ADX14', 'KOSPI 200_BB_width', 'KOSPI 200_PPO', 'KOSPI 200_PPO_Hist', 'KOSPI 200_OG', 'GJR_VaR_5_t1', 'Signal1_Buy', 'Signal1_Sell', 'Signal2_Buy', 'Signal2_Sell', 'Signal3_Buy', 'Signal3_Sell', 'Signal4_Buy', 'Signal4_Sell']


In [7]:
# =========================
# 6. Scaling & discrete feature mask
#    mutual information 계산 안정성을 위해 train 기준 MinMax scaling
# =========================
scaler = MinMaxScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_names
)

X_valid_scaled = pd.DataFrame(
    scaler.transform(X_valid),
    columns=feature_names
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_names
)

# binary/dummy 변수 판별
# 0/1 더미는 discrete로 처리하고, 나머지는 continuous로 처리

def is_binary_series(s):
    vals = pd.Series(s).dropna().unique()
    return len(vals) <= 2 and set(vals).issubset({0, 1})

# 원자료 X_train 기준으로 binary 여부 판별
discrete_mask = np.array([
    is_binary_series(X_train[col]) for col in feature_names
])

print("Discrete/binary feature count:", int(discrete_mask.sum()))
print("Continuous feature count:", int((~discrete_mask).sum()))


Discrete/binary feature count: 8
Continuous feature count: 20


In [8]:
# =========================
# 7. Relevance 계산
#    relevance = I(X_j ; y)
# =========================
relevance_values = mutual_info_classif(
    X_train_scaled,
    y_train,
    discrete_features=discrete_mask,
    random_state=RANDOM_STATE
)

relevance = pd.Series(relevance_values, index=feature_names)

relevance_df = (
    relevance
    .reset_index()
    .rename(columns={"index": "feature", 0: "relevance_MI_with_target"})
    .sort_values("relevance_MI_with_target", ascending=False)
    .reset_index(drop=True)
)

print("=" * 80)
print("전체 변수 relevance: mutual information with Risk_Label")
print("=" * 80)
display(relevance_df)


전체 변수 relevance: mutual information with Risk_Label


,feature,relevance_MI_with_target
0,NASDAQ_return(%),0.039116
1,VKOSPI_Close,0.011601
2,KOSPI 200_OG,0.009234
3,Brent Crude Oil_return(%),0.009192
4,KOSPI 200_DMI14,0.009141
5,Gold Spot_return(%),0.007848
6,GJR_VaR_5_t1,0.003829
7,KOSPI 200 lagged_1_return(%),0.003665
8,KOSPI 200 Volume,0.003466
9,KOSPI 200_PPO_Hist,0.003132


In [9]:
# =========================
# 8. mRMR helper functions
#    redundancy = 평균 I(X_j ; X_s), X_s는 이미 선택된 변수
# =========================
def mutual_info_between_features(candidate_feature, selected_feature):
    """
    candidate_feature와 selected_feature 간 mutual information 계산.
    selected_feature가 binary면 classification MI,
    continuous면 regression MI를 사용.
    """
    x_candidate = X_train_scaled[[candidate_feature]]

    selected_idx = feature_names.index(selected_feature)
    candidate_idx = feature_names.index(candidate_feature)

    selected_is_discrete = discrete_mask[selected_idx]
    candidate_is_discrete = np.array([discrete_mask[candidate_idx]])

    if selected_is_discrete:
        y_selected = X_train[selected_feature].astype(int)
        mi = mutual_info_classif(
            x_candidate,
            y_selected,
            discrete_features=candidate_is_discrete,
            random_state=RANDOM_STATE
        )[0]
    else:
        y_selected = X_train_scaled[selected_feature]
        mi = mutual_info_regression(
            x_candidate,
            y_selected,
            discrete_features=candidate_is_discrete,
            random_state=RANDOM_STATE
        )[0]

    return float(mi)


In [10]:
# =========================
# 9. mRMR Forward Selection
#    score = relevance - redundancy
# =========================
selected_features_mrmr = []
remaining_features = feature_names.copy()
selection_history = []

for step in range(1, TOP_K + 1):
    candidate_rows = []

    for feat in remaining_features:
        rel = float(relevance[feat])

        if len(selected_features_mrmr) == 0:
            red = 0.0
        else:
            red_list = [
                mutual_info_between_features(feat, selected_feat)
                for selected_feat in selected_features_mrmr
            ]
            red = float(np.mean(red_list))

        mrmr_score = rel - red

        candidate_rows.append({
            "feature": feat,
            "step": step,
            "relevance": rel,
            "redundancy": red,
            "mrmr_score": float(mrmr_score)
        })

    candidate_df = pd.DataFrame(candidate_rows)

    # mRMR score 우선, 동점이면 target relevance 큰 변수 우선
    best_row = candidate_df.sort_values(
        ["mrmr_score", "relevance"],
        ascending=[False, False]
    ).iloc[0]

    best_feature = best_row["feature"]

    selected_features_mrmr.append(best_feature)
    remaining_features.remove(best_feature)

    selection_history.append({
        "selection_order": step,
        "feature": best_feature,
        "relevance": float(best_row["relevance"]),
        "redundancy": float(best_row["redundancy"]),
        "mrmr_score": float(best_row["mrmr_score"])
    })

mrmr_selected_df = pd.DataFrame(selection_history)

print("=" * 80)
print("mRMR selection history")
print("=" * 80)
display(mrmr_selected_df)


# =========================
# Signal 계열 후처리
# - Signal족 변수(Buy/Sell/Stay) 중 하나라도 mRMR에서 선택되면
#   최종 모델 입력 변수에는 해당 Signal 그룹의 Buy/Sell을 둘 다 포함
# - Stay는 기준범주로 보고 최종 변수에서는 제외
# =========================
import re

selected_features_mrmr_raw = selected_features_mrmr.copy()

signal_pattern = re.compile(r"^(Signal\d+)_(Buy|Sell|Stay)$")


def expand_signal_buy_sell(selected_features, all_features):
    all_features_set = set(all_features)
    expanded_features = []
    expanded_signal_groups = []

    for feature in selected_features:
        match = signal_pattern.match(feature)

        if match:
            sig = match.group(1)
            expanded_signal_groups.append(sig)

            buy_col = f"{sig}_Buy"
            sell_col = f"{sig}_Sell"

            if buy_col in all_features_set:
                expanded_features.append(buy_col)
            if sell_col in all_features_set:
                expanded_features.append(sell_col)
        else:
            expanded_features.append(feature)

    # 중복 제거하면서 순서 유지
    expanded_features = list(dict.fromkeys(expanded_features))
    expanded_signal_groups = sorted(set(expanded_signal_groups))

    return expanded_features, expanded_signal_groups


selected_features_mrmr, expanded_signal_groups = expand_signal_buy_sell(
    selected_features_mrmr_raw,
    feature_names
)

print("\n" + "=" * 80)
print("Signal Buy/Sell pair expansion")
print("=" * 80)
print(f"mRMR 원 선택 변수 개수: {len(selected_features_mrmr_raw)}")
print(f"Signal 확장 후 최종 변수 개수: {len(selected_features_mrmr)}")
print(f"Buy/Sell 동시 포함 처리된 Signal 그룹: {expanded_signal_groups}")
print("\n[원 mRMR 선택 변수]")
print(selected_features_mrmr_raw)
print("\n[Signal 처리 후 최종 선택 변수]")
print(selected_features_mrmr)


mRMR selection history


,selection_order,feature,relevance,redundancy,mrmr_score
0,1,NASDAQ_return(%),0.039116,0.000000,0.039116
1,2,KOSPI 200_DMI14,0.009141,0.000000,0.009141
2,3,Signal1_Buy,0.000351,0.005876,-0.005525
3,4,Signal1_Sell,0.000028,0.006092,-0.006064
4,5,Signal3_Buy,0.000019,0.005411,-0.005393
5,6,Signal4_Sell,0.001003,0.005799,-0.004796
6,7,Signal3_Sell,0.000001,0.004507,-0.004506
7,8,KOSPI 200_OG,0.009234,0.012347,-0.003113
8,9,Signal4_Buy,0.000057,0.004107,-0.004050
9,10,Gold Spot_return(%),0.007848,0.010942,-0.003094



Signal Buy/Sell pair expansion
mRMR 원 선택 변수 개수: 12
Signal 확장 후 최종 변수 개수: 13
Buy/Sell 동시 포함 처리된 Signal 그룹: ['Signal1', 'Signal2', 'Signal3', 'Signal4']

[원 mRMR 선택 변수]
['NASDAQ_return(%)', 'KOSPI 200_DMI14', 'Signal1_Buy', 'Signal1_Sell', 'Signal3_Buy', 'Signal4_Sell', 'Signal3_Sell', 'KOSPI 200_OG', 'Signal4_Buy', 'Gold Spot_return(%)', 'KOSPI 200 Volume', 'Signal2_Buy']

[Signal 처리 후 최종 선택 변수]
['NASDAQ_return(%)', 'KOSPI 200_DMI14', 'Signal1_Buy', 'Signal1_Sell', 'Signal3_Buy', 'Signal3_Sell', 'Signal4_Buy', 'Signal4_Sell', 'KOSPI 200_OG', 'Gold Spot_return(%)', 'KOSPI 200 Volume', 'Signal2_Buy', 'Signal2_Sell']


In [11]:
# =========================
# 10. 전체 변수표: 선택 여부 포함
#     Signal족은 하나라도 선택되면 Buy/Sell 둘 다 최종 선택으로 반영
# =========================
mrmr_full_df = relevance_df.copy()

# 최종 선택 여부: Signal Buy/Sell 확장 후 변수 기준
mrmr_full_df["selected_by_mrmr"] = mrmr_full_df["feature"].isin(selected_features_mrmr).astype(int)

# 원 mRMR 선택 여부와 Signal 후처리로 추가된 변수 구분
if "selected_features_mrmr_raw" in globals():
    mrmr_full_df["selected_raw_mrmr"] = mrmr_full_df["feature"].isin(selected_features_mrmr_raw).astype(int)
    mrmr_full_df["added_by_signal_pair_rule"] = (
        (mrmr_full_df["selected_by_mrmr"] == 1) &
        (mrmr_full_df["selected_raw_mrmr"] == 0)
    ).astype(int)
else:
    mrmr_full_df["selected_raw_mrmr"] = mrmr_full_df["selected_by_mrmr"]
    mrmr_full_df["added_by_signal_pair_rule"] = 0

order_map = {f: i + 1 for i, f in enumerate(selected_features_mrmr)}
mrmr_full_df["selection_order"] = mrmr_full_df["feature"].map(order_map)

mrmr_full_df = mrmr_full_df.sort_values(
    ["selected_by_mrmr", "selection_order", "relevance_MI_with_target"],
    ascending=[False, True, False]
).reset_index(drop=True)

print("=" * 80)
print("mRMR full feature table")
print("=" * 80)
display(mrmr_full_df)

# 이후 최종 모델 입력용 데이터셋
X_train_mrmr = X_train[selected_features_mrmr].copy()
X_valid_mrmr = X_valid[selected_features_mrmr].copy()
X_test_mrmr  = X_test[selected_features_mrmr].copy()

print("\nFinal data shapes")
print("X_train_mrmr:", X_train_mrmr.shape)
print("X_valid_mrmr:", X_valid_mrmr.shape)
print("X_test_mrmr :", X_test_mrmr.shape)


mRMR full feature table


,feature,relevance_MI_with_target,selected_by_mrmr,selected_raw_mrmr,added_by_signal_pair_rule,selection_order
0,NASDAQ_return(%),0.039116,1,1,0,1.0
1,KOSPI 200_DMI14,0.009141,1,1,0,2.0
2,Signal1_Buy,0.000351,1,1,0,3.0
3,Signal1_Sell,0.000028,1,1,0,4.0
4,Signal3_Buy,0.000019,1,1,0,5.0
5,Signal3_Sell,0.000001,1,1,0,6.0
6,Signal4_Buy,0.000057,1,1,0,7.0
7,Signal4_Sell,0.001003,1,1,0,8.0
8,KOSPI 200_OG,0.009234,1,1,0,9.0
9,Gold Spot_return(%),0.007848,1,1,0,10.0



Final data shapes
X_train_mrmr: (1848, 13)
X_valid_mrmr: (1438, 13)
X_test_mrmr : (822, 13)


In [12]:
# =========================
# 11. 최종 선택 피처 리스트 출력
#     이후 모델 코드에 바로 복붙할 수 있도록 [ ] 형태로 출력
# =========================

print("mRMR selected features")
print("=" * 80)
print(f"TOP_K = {TOP_K}")

if "selected_features_mrmr_raw" in globals():
    print(f"원 mRMR 선택 변수 개수: {len(selected_features_mrmr_raw)}")
    print("원 mRMR 선택 변수:")
    print(selected_features_mrmr_raw)
    print("-" * 80)

print(f"Signal 처리 후 최종 남길 변수 개수: {len(selected_features_mrmr)}")
print(selected_features_mrmr)


mRMR selected features
TOP_K = 12
원 mRMR 선택 변수 개수: 12
원 mRMR 선택 변수:
['NASDAQ_return(%)', 'KOSPI 200_DMI14', 'Signal1_Buy', 'Signal1_Sell', 'Signal3_Buy', 'Signal4_Sell', 'Signal3_Sell', 'KOSPI 200_OG', 'Signal4_Buy', 'Gold Spot_return(%)', 'KOSPI 200 Volume', 'Signal2_Buy']
--------------------------------------------------------------------------------
Signal 처리 후 최종 남길 변수 개수: 13
['NASDAQ_return(%)', 'KOSPI 200_DMI14', 'Signal1_Buy', 'Signal1_Sell', 'Signal3_Buy', 'Signal3_Sell', 'Signal4_Buy', 'Signal4_Sell', 'KOSPI 200_OG', 'Gold Spot_return(%)', 'KOSPI 200 Volume', 'Signal2_Buy', 'Signal2_Sell']
